# certified-dose: Quickstart Guide

Welcome to **`certified-dose`**! This tutorial demonstrates how to use formal reachability analysis to guarantee regulatory compliance in process-control chemical dosing under bounded input uncertainty.

> **Notice**: The process model used here is synthetic and intended for demonstration and research.

In [ ]:
import numpy as np
from certified_dose import (
    Interval,
    AffineForm,
    SyntheticProcessModel,
    ReachabilityEngine,
    CertifiedDoseWrapper,
    AggressiveCostMinimizerController,
    PlantState,
    simulate_closed_loop,
)

print("certified-dose successfully imported!")

## 1. Rigorous Interval Arithmetic

Standard floating-point numbers represent single points. The `Interval` primitive represents closed sets $[\underline{x}, \overline{x}]$ with conservative over-approximation guarantees.

In [ ]:
# Define uncertain sensor readings
turbidity = Interval(20.0, 26.0)  # Raw turbidity: 23 +/- 3 NTU
flow = Interval(950.0, 1050.0)    # Flow rate: 1000 +/- 50 m3/h

print(f"Turbidity interval: {turbidity}")
print(f"Flow interval: {flow}")
print(f"Addition: {turbidity + 5.0}")
print(f"Multiplication: {turbidity * flow}")

## 2. Process Model and Reachability Engine

The `SyntheticProcessModel` models nonlinear coagulant kinetics (colloidal removal vs overdosing restabilization).

In [ ]:
model = SyntheticProcessModel()
engine = ReachabilityEngine(model=model, safety_margin=0.02)

# Evaluate reachable effluent turbidity for a candidate dose of 15.0 mg/L
disturbances = {
    "turbidity": Interval(22.0, 28.0),
    "flow_rate": Interval(900.0, 1100.0),
    "ph": Interval(7.1, 7.5),
    "temperature": Interval(16.0, 18.0),
}

reachable = engine.compute_reachable_set(dose=15.0, disturbances=disturbances)
print(f"Dose: {reachable.dose} mg/L")
print(f"Guaranteed Reachable Effluent: [{reachable.lo:.3f}, {reachable.hi:.3f}] NTU")

## 3. Fuzzing Conservativeness Against Monte Carlo Sampling

We verify that our interval bounds enclose 1,000 random Monte Carlo evaluations.

In [ ]:
mc_result = engine.verify_against_monte_carlo(dose=15.0, disturbances=disturbances, n_samples=1000)
print(f"Conservativeness sound? {mc_result['is_conservative']}")
print(f"Monte Carlo Range: [{mc_result['mc_min']:.3f}, {mc_result['mc_max']:.3f}] NTU")
print(f"Reachable Interval: [{mc_result['reachable_lo']:.3f}, {mc_result['reachable_hi']:.3f}] NTU")
print(f"Safety Margin on Upper Bound: +{mc_result['margin_hi']:.4f} NTU")

## 4. Formal Safety Certifier Wrapper

The `CertifiedDoseWrapper` enforces that the upper reachable bound never exceeds the compliance ceiling (e.g. 1.0 NTU).

In [ ]:
certifier = CertifiedDoseWrapper(
    engine=engine,
    compliance_limit=1.0,  # 1.0 NTU limit
    fallback_dose=24.0,
)

# Test 1: Aggressive under-dose (e.g. 8.0 mg/L proposed by an RL agent)
res_unsafe = certifier.certify_action(proposed_dose=8.0, disturbances=disturbances)
print(f"Unsafe Action Status: {res_unsafe.status.value}")
print(f"Corrected Dose: {res_unsafe.certified_dose:.2f} mg/L")
print(f"Reason: {res_unsafe.reason}\n")

# Test 2: Safe adequate dose (e.g. 22.0 mg/L)
res_safe = certifier.certify_action(proposed_dose=22.0, disturbances=disturbances)
print(f"Safe Action Status: {res_safe.status.value}")
print(f"Certified Dose: {res_safe.certified_dose:.2f} mg/L")

## 5. End-to-End Simulation Benchmark

Run a 100-step dynamic closed loop simulation comparing the unverified candidate controller against the formally guarded plant.

In [ ]:
summary = simulate_closed_loop(n_steps=100, seed=42, compliance_limit=1.0)

print(f"Total Steps: {summary.total_steps}")
print(f"Unchecked Controller Violations: {summary.candidate_violations}")
print(f"Certified Plant Violations: {summary.certified_violations} (Zero!)")
print(f"Safety Wrapper Interventions: {summary.interventions} ({summary.intervention_rate_pct:.1f}%)")